# 3.3 线性回归的简洁实现 (Concise Implementation of Linear Regression)
> 对应教材《动手学深度学习》(PyTorch版) 第3.3节。  
> 使用 PyTorch 高层 API（`torch.utils.data`、`torch.nn`、`torch.optim`）重构线性回归训练流水线，并完成 3 道课后练习题。


In [1]:
# 【所属小节】3.3.1 生成数据集
# 【作用说明】导入 NumPy、PyTorch 数据处理模块以及 d2l 工具包，生成 1000 个带噪声的二维线性样本
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)


In [2]:
# 【所属小节】3.3.2 读取数据集
# 【作用说明】利用 TensorDataset 打包张量，通过 DataLoader 构造支持批处理与自动打乱的数据迭代器
def load_array(data_arrays, batch_size, is_train=True):
    #@save
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

# 读取并打印第一个小批量样本验证
next(iter(data_iter))


[tensor([[-0.5574, -0.0261],
         [ 1.4029,  1.4674],
         [ 0.7196,  0.1711],
         [-1.3968,  0.1471],
         [ 0.4205, -0.9768],
         [-0.7285, -0.0139],
         [ 0.8362, -0.2483],
         [ 0.5906,  1.3400],
         [-0.6626, -0.1495],
         [ 0.7977, -0.6351]]),
 tensor([[3.1748],
         [2.0126],
         [5.0359],
         [0.9056],
         [8.3551],
         [2.7977],
         [6.7282],
         [0.8384],
         [3.3753],
         [7.9309]])]

In [3]:
# 【所属小节】3.3.3 定义模型
# 【作用说明】使用 nn.Sequential 容器容纳单层全连接网络 nn.Linear(in_features=2, out_features=1)
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))


In [4]:
# 【所属小节】3.3.4 初始化模型参数
# 【作用说明】通过 net[0] 访问全连接层，直接用 normal_ 将权重覆盖为正态分布，用 fill_ 将偏置填充为 0
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)


tensor([0.])

In [5]:
# 【所属小节】3.3.5 & 3.3.6 定义损失函数与优化算法
# 【作用说明】实例化均方误差损失函数 nn.MSELoss() 与小批量随机梯度下降优化器 torch.optim.SGD
loss = nn.MSELoss()
trainer = torch.optim.SGD(net.parameters(), lr=0.03)


In [6]:
# 【所属小节】3.3.7 训练
# 【作用说明】执行标准化工业级训练循环：前向传播 -> 梯度清零 -> 反向传播 -> 优化器单步更新 -> 周期评估
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

# 比较真实参数与训练学得参数
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)


epoch 1, loss 0.000364
epoch 2, loss 0.000104
epoch 3, loss 0.000104
w的估计误差： tensor([0.0002, 0.0006])
b的估计误差： tensor([0.0008])


---
## 课后练习题实验与算法验证
包含第 3.3 节全部 3 道课后练习题的实现与验证。


In [7]:
# 【课后练习 1, 2, 3】
# 1. 练习 1: 平均损失 vs 总损失学习率对应关系验证
# 在 nn.MSELoss() 中默认计算平均损失，trainer 学习率保持 lr=0.03 即可与手工 SGD 保持完全等价步长

# 2. 练习 2: 使用 Huber 损失 (SmoothL1Loss) 训练模型
net_huber = nn.Sequential(nn.Linear(2, 1))
net_huber[0].weight.data.normal_(0, 0.01)
net_huber[0].bias.data.fill_(0)

# 使用 Huber 损失代替 MSE
huber_loss = nn.HuberLoss(delta=0.01)
trainer_huber = torch.optim.SGD(net_huber.parameters(), lr=0.03)

for epoch in range(3):
    for X, y in data_iter:
        l = huber_loss(net_huber(X), y)
        trainer_huber.zero_grad()
        l.backward()
        trainer_huber.step()
    print(f'[Huber Loss] epoch {epoch + 1}, loss: {huber_loss(net_huber(features), labels):.6f}')

w_huber = net_huber[0].weight.data
print('Huber 学得权重误差:', true_w - w_huber.reshape(true_w.shape))

# 3. 练习 3: 访问线性回归模型的参数梯度
# 在反向传播后直接访问 .grad 属性
print('\n【练习 3 验证】当前批次 weight 梯度:\n', net_huber[0].weight.grad)
print('【练习 3 验证】当前批次 bias 梯度:\n', net_huber[0].bias.grad)


[Huber Loss] epoch 1, loss: 0.045590
[Huber Loss] epoch 2, loss: 0.045389
[Huber Loss] epoch 3, loss: 0.045189
Huber 学得权重误差: tensor([ 1.9892, -3.3772])

【练习 3 验证】当前批次 weight 梯度:
 tensor([[-0.0033,  0.0032]])
【练习 3 验证】当前批次 bias 梯度:
 tensor([-0.0080])
